## License
Copyright 2026 jphall@gwu.edu. MIT License; see the repository LICENSE file.

# Cited RAG over NIST AI RMF actions

Retrieval-augmented generation grounds an answer in local sources: search, retrieve, augment, generate, and cite. The similarity calculation and citations are made explicit, rather than hidden behind a framework.

Run `embedding_example/get_embeddings.ipynb` first. It produces `data/rmf_action_embeddings.csv` from the NIST AI RMF actions.


## 1. Import packages and load the vector dataset


In [8]:
# 1. Import packages and load the vector dataset
# Load the embedding output created by the embedding notebook.

from pathlib import Path
import json, os
from getpass import getpass
import numpy as np
import pandas as pd
from openai import AzureOpenAI

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists(): ROOT = ROOT.parent

documents = pd.read_csv(ROOT / "data" / "rmf_action_embeddings.csv")
vectors = np.vstack(documents.embedding.map(json.loads))

RESOURCE = "gw-sb-01"
ENDPOINT = f"https://{RESOURCE}.openai.azure.com/"

api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")
client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=api_key, api_version="2025-03-01-preview")


## 2. Embed and retrieve for a question


In [9]:
# 2. Embed and retrieve for a question
# Use cosine similarity to identify the most relevant actions.

question = "How can an organization make people accountable for AI risk management?"
query = np.array(client.embeddings.create(model="text-embedding-3-small", input=[question]).data[0].embedding)

print(f"Embedded query: {query}")
print(f"Embedded query length: {len(query)}")
print()

# The @ operator calculates one dot product between the query and every document vector.
# Dividing by both vector lengths normalizes those dot products into cosine similarity scores.
# Scores near 1 mean the query and action point in similar semantic directions.
scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))

retrieved = documents.assign(similarity=scores).nlargest(4, "similarity")

display(retrieved[["control_id", "function", "action", "similarity"]])


Embedded query: [ 0.03109741  0.02938843  0.09082031 ... -0.02362061 -0.00192261
  0.00272369]
Embedded query length: 1536



,control_id,function,action,similarity
910,MS-2.8-020,MEASURE,Track and audit the effectiveness of organizat...,0.777767
8,GV-2,GOVERN,Accountability structures are in place so that...,0.774739
177,GV-2.3-001,GOVERN,Organizational management can: Declare risk to...,0.752424
15,GV-4,GOVERN,Organizational teams are committed to a cultur...,0.735186


## 3. Generate a cited answer


In [10]:
# 3. Generate a cited answer
# Give the model retrieved context and print the source controls.

context = "\n\n".join(f"[{row.control_id}] {row.action}" for row in retrieved.itertuples())
prompt = f"Answer using only these retrieved actions. Cite every claim with its bracketed control ID. Do not make uncitable claims or additional responses. \n\nQuestion: {question}\n\nSources:\n{context}"

print("RAG prompt:")
print(prompt)
print()
print("=========== ==========")

# GPT-5 uses tokens for internal reasoning; leave room for the visible cited answer.
response = client.responses.create(
    model="gpt-5-mini",
    input=prompt,
    max_output_tokens=1600,
    reasoning={"effort": "minimal"},
)
answer = response.output_text.strip()

# Do not present an empty response as though it were a grounded answer.
if not answer:
    raise RuntimeError("The LLM returned no visible answer. Please run this question again.")
print()
print("LLM Response:")
print(answer)
print()
print("\nRetrieved sources:")
for row in retrieved.itertuples(): print(f"[{row.control_id}] {row.action}")


RAG prompt:
Answer using only these retrieved actions. Cite every claim with its bracketed control ID. Do not make uncitable claims or additional responses. 

Question: How can an organization make people accountable for AI risk management?

Sources:
[MS-2.8-020] Track and audit the effectiveness of organizational mechanisms related to AI risk management, including: Lines of communication between AI actors, executive leadership, users and impacted communities; Roles and responsibilities for AI actors and executive leadership; Organizational accountability roles, e.g., chief model risk officers, AI oversight committees, responsible or ethical AI directors, etc.

[GV-2] Accountability structures are in place so that the appropriate teams and individuals are empowered, responsible, and trained for mapping, measuring, and managing AI risks.

[GV-2.3-001] Organizational management can: Declare risk tolerances for developing or using AI systems; Support AI risk management efforts, and play a